In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/pill_detection'
DATA_DIR = f'{BASE}/data'
TRAIN_IMG_DIR = f'{DATA_DIR}/train_images'
TRAIN_ANN_DIR = f'{DATA_DIR}/train_annotations'
TEST_IMG_DIR = f'{DATA_DIR}/test_images'
OUTPUT_DIR = f'{BASE}/outputs'
YOLO_DIR = f'{BASE}/yolo_data'
yaml_path = f'{YOLO_DIR}/data.yaml'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

!pip install ultralytics ensemble-boxes -q

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.7 MB/s eta 0:00:00


In [ ]:
import glob, json, csv, os
import pandas as pd
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion

# category_id 매핑
json_files = glob.glob(f'{TRAIN_ANN_DIR}/**/*.json', recursive=True)
dlid_set = set()
for jf in json_files:
    with open(jf, 'r', encoding='utf-8') as f:
        data = json.load(f)
    for img_info in data['images']:
        dlid_set.add(int(img_info['dl_idx']))

dlid_to_idx = {dlid: i+1 for i, dlid in enumerate(sorted(dlid_set))}
idx_to_catid = {v: k for k, v in dlid_to_idx.items()}
print(f"매핑 완료: {len(idx_to_catid)}개")

# 예측 함수
def predict_with_tta(model_path, model_name):
    model = YOLO(model_path)
    test_imgs = sorted([f for f in os.listdir(TEST_IMG_DIR) if f.endswith('.png')])
    rows = []
    ann_id = 1
    for img_file in test_imgs:
        image_id = int(img_file.replace('.png', ''))
        img_path = os.path.join(TEST_IMG_DIR, img_file)
        results = model.predict(img_path, conf=0.25, iou=0.5, augment=True, verbose=False)
        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                w, h = x2 - x1, y2 - y1
                score = float(box.conf[0])
                cat_id = idx_to_catid[int(box.cls[0]) + 1]
                rows.append([ann_id, image_id, cat_id,
                              round(x1,2), round(y1,2),
                              round(w,2), round(h,2),
                              round(score,4)])
                ann_id += 1
    out_path = f'{OUTPUT_DIR}/pred_{model_name}.csv'
    with open(out_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['annotation_id','image_id','category_id',
                         'bbox_x','bbox_y','bbox_w','bbox_h','score'])
        writer.writerows(rows)
    print(f"[{model_name}] {len(rows)} rows → {out_path}")
    return out_path

# WBF 앙상블 함수
def run_wbf_ensemble(pred_paths, weights, iou_thr=0.55, skip_box_thr=0.3, out_name='ensemble'):
    dfs = [pd.read_csv(p) for p in pred_paths]
    all_image_ids = sorted(set().union(*[set(df['image_id']) for df in dfs]))
    IMG_W, IMG_H = 976, 1280
    rows = []
    ann_id = 1
    for img_id in all_image_ids:
        boxes_list, scores_list, labels_list = [], [], []
        for df in dfs:
            sub = df[df['image_id'] == img_id]
            if len(sub) == 0:
                boxes_list.append([]); scores_list.append([]); labels_list.append([])
                continue
            bboxes = []
            for _, row in sub.iterrows():
                x1 = row['bbox_x'] / IMG_W
                y1 = row['bbox_y'] / IMG_H
                x2 = (row['bbox_x'] + row['bbox_w']) / IMG_W
                y2 = (row['bbox_y'] + row['bbox_h']) / IMG_H
                bboxes.append([max(0,min(1,x1)), max(0,min(1,y1)),
                                max(0,min(1,x2)), max(0,min(1,y2))])
            boxes_list.append(bboxes)
            scores_list.append(sub['score'].tolist())
            labels_list.append(sub['category_id'].tolist())
        boxes, scores, labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            weights=weights, iou_thr=iou_thr, skip_box_thr=skip_box_thr
        )
        for box, score, label in zip(boxes, scores, labels):
            x1 = box[0] * IMG_W
            y1 = box[1] * IMG_H
            w  = (box[2] - box[0]) * IMG_W
            h  = (box[3] - box[1]) * IMG_H
            rows.append([ann_id, img_id, int(label),
                         round(x1,2), round(y1,2),
                         round(w,2), round(h,2),
                         round(float(score),4)])
            ann_id += 1
    out_path = f'{OUTPUT_DIR}/{out_name}.csv'
    with open(out_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['annotation_id','image_id','category_id',
                         'bbox_x','bbox_y','bbox_w','bbox_h','score'])
        writer.writerows(rows)
    print(f"[{out_name}] {len(rows)} rows → {out_path}")
    return out_path

print("세팅 완료!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
매핑 완료: 56개
세팅 완료!


In [ ]:
# 기존 예측 CSV 경로 로드
pred_8s  = f'{OUTPUT_DIR}/pred_v8s.csv'
pred_8m  = f'{OUTPUT_DIR}/pred_v8m.csv'
pred_11s = f'{OUTPUT_DIR}/pred_v11s.csv'

# 파일 존재 확인
import os
for name, path in [('v8s', pred_8s), ('v8m', pred_8m), ('v11s', pred_11s)]:
    exists = '✅' if os.path.exists(path) else '❌'
    print(f"{exists} {name}: {path}")

✅ v8s: /content/drive/MyDrive/pill_detection/outputs/pred_v8s.csv
✅ v8m: /content/drive/MyDrive/pill_detection/outputs/pred_v8m.csv
✅ v11s: /content/drive/MyDrive/pill_detection/outputs/pred_v11s.csv


In [ ]:
pred_paths_3 = [pred_8s, pred_8m, pred_11s]

# iou_thr / skip_box_thr 튜닝 실험
tuning_experiments = {
    'iou45':  (0.45, 0.3),
    'iou50':  (0.50, 0.3),
    'iou60':  (0.60, 0.3),
    'skip20': (0.55, 0.2),
    'skip40': (0.55, 0.4),
}

for name, (iou, skip) in tuning_experiments.items():
    run_wbf_ensemble(
        pred_paths_3,
        weights=[1.0, 1.0, 1.0],  # equal이 최고점이었으니 균등 가중치 유지
        iou_thr=iou,
        skip_box_thr=skip,
        out_name=f'wbf_tuned_{name}'
    )

[wbf_tuned_iou45] 4044 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_tuned_iou45.csv
[wbf_tuned_iou50] 4044 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_tuned_iou50.csv
[wbf_tuned_iou60] 4044 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_tuned_iou60.csv
[wbf_tuned_skip20] 4160 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_tuned_skip20.csv
[wbf_tuned_skip40] 3857 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_tuned_skip40.csv


In [ ]:
import pandas as pd
df = pd.read_csv(f'{OUTPUT_DIR}/wbf_tuned_iou50.csv')
print(df['category_id'].head(5).tolist())

[39, 1, 17, 42, 37]


In [ ]:
def run_wbf_ensemble(pred_paths, weights, iou_thr=0.55, skip_box_thr=0.3, out_name='ensemble'):
    dfs = [pd.read_csv(p) for p in pred_paths]
    all_image_ids = sorted(set().union(*[set(df['image_id']) for df in dfs]))
    IMG_W, IMG_H = 976, 1280
    rows = []
    ann_id = 1
    for img_id in all_image_ids:
        boxes_list, scores_list, labels_list = [], [], []
        for df in dfs:
            sub = df[df['image_id'] == img_id]
            if len(sub) == 0:
                boxes_list.append([]); scores_list.append([]); labels_list.append([])
                continue
            bboxes = []
            for _, row in sub.iterrows():
                x1 = row['bbox_x'] / IMG_W
                y1 = row['bbox_y'] / IMG_H
                x2 = (row['bbox_x'] + row['bbox_w']) / IMG_W
                y2 = (row['bbox_y'] + row['bbox_h']) / IMG_H
                bboxes.append([max(0,min(1,x1)), max(0,min(1,y1)),
                                max(0,min(1,x2)), max(0,min(1,y2))])
            boxes_list.append(bboxes)
            scores_list.append(sub['score'].tolist())
            labels_list.append(sub['category_id'].tolist())
        boxes, scores, labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            weights=weights, iou_thr=iou_thr, skip_box_thr=skip_box_thr
        )
        for box, score, label in zip(boxes, scores, labels):
            x1 = box[0] * IMG_W
            y1 = box[1] * IMG_H
            w  = (box[2] - box[0]) * IMG_W
            h  = (box[3] - box[1]) * IMG_H
            rows.append([ann_id, img_id, int(label),  # ← 이미 dl_idx라 변환 불필요
                         round(x1,2), round(y1,2),
                         round(w,2), round(h,2),
                         round(float(score),4)])
            ann_id += 1
    out_path = f'{OUTPUT_DIR}/{out_name}.csv'
    with open(out_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['annotation_id','image_id','category_id',
                         'bbox_x','bbox_y','bbox_w','bbox_h','score'])
        writer.writerows(rows)
    print(f"[{out_name}] {len(rows)} rows → {out_path}")
    return out_path

In [ ]:
import pandas as pd

pred_files = ['pred_v8s', 'pred_v8m', 'pred_v11s', 'pred_v11s_aug', 'pred_v11m']

for name in pred_files:
    path = f'{OUTPUT_DIR}/{name}.csv'
    df = pd.read_csv(path)
    df['category_id'] = df['category_id'].apply(lambda x: idx_to_catid[x])
    df.to_csv(path, index=False)
    print(f"[{name}] 변환 완료 → 샘플: {df['category_id'].head(3).tolist()}")

[pred_v8s] 변환 완료 → 샘플: [27926, 16551, 1900]
[pred_v8m] 변환 완료 → 샘플: [27926, 1900, 16551]
[pred_v11s] 변환 완료 → 샘플: [27926, 1900, 16551]
[pred_v11s_aug] 변환 완료 → 샘플: [16551, 27926, 1900]
[pred_v11m] 변환 완료 → 샘플: [27926, 1900, 16551]


In [ ]:
pred_paths_3 = [pred_8s, pred_8m, pred_11s]

tuning_experiments = {
    'iou45':  (0.45, 0.3),
    'iou50':  (0.50, 0.3),
    'iou60':  (0.60, 0.3),
    'skip20': (0.55, 0.2),
    'skip40': (0.55, 0.4),
}

for name, (iou, skip) in tuning_experiments.items():
    run_wbf_ensemble(
        pred_paths_3,
        weights=[1.0, 1.0, 1.0],
        iou_thr=iou,
        skip_box_thr=skip,
        out_name=f'wbf_tuned_{name}'
    )

[wbf_tuned_iou45] 4044 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_tuned_iou45.csv
[wbf_tuned_iou50] 4044 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_tuned_iou50.csv
[wbf_tuned_iou60] 4044 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_tuned_iou60.csv
[wbf_tuned_skip20] 4160 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_tuned_skip20.csv
[wbf_tuned_skip40] 3857 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_tuned_skip40.csv


In [ ]:
df = pd.read_csv(f'{OUTPUT_DIR}/wbf_tuned_iou50.csv')
print(df['category_id'].head(5).tolist())

[27926, 1900, 16551, 29345, 27733]


In [ ]:
day4_experiments = {
    'iou45':          (0.45, 0.30),
    'iou60':          (0.60, 0.30),
    'skip15':         (0.55, 0.15),
    'skip10':         (0.55, 0.10),
    'skip20_iou50':   (0.50, 0.20),
    'skip20_iou45':   (0.45, 0.20),
    'skip15_iou50':   (0.50, 0.15),
    'skip15_iou45':   (0.45, 0.15),
    'skip10_iou50':   (0.50, 0.10),
    'skip10_iou45':   (0.45, 0.10),
}

for name, (iou, skip) in day4_experiments.items():
    run_wbf_ensemble(
        pred_paths_3,
        weights=[1.0, 1.0, 1.0],
        iou_thr=iou,
        skip_box_thr=skip,
        out_name=f'wbf_d4_{name}'
    )

[wbf_d4_iou45] 4044 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_d4_iou45.csv
[wbf_d4_iou60] 4044 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_d4_iou60.csv
[wbf_d4_skip15] 4160 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_d4_skip15.csv
[wbf_d4_skip10] 4160 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_d4_skip10.csv
[wbf_d4_skip20_iou50] 4160 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_d4_skip20_iou50.csv
[wbf_d4_skip20_iou45] 4160 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_d4_skip20_iou45.csv
[wbf_d4_skip15_iou50] 4160 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_d4_skip15_iou50.csv
[wbf_d4_skip15_iou45] 4160 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_d4_skip15_iou45.csv
[wbf_d4_skip10_iou50] 4160 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_d4_skip10_iou50.csv
[wbf_d4_skip10_iou45] 4160 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_d4_skip10_iou45.csv


In [ ]:
df = pd.read_csv(f'{OUTPUT_DIR}/wbf_d4_skip15.csv')
print(df['category_id'].head(5).tolist())

[27926, 1900, 16551, 29345, 27733]
